# Li2026 (Triple-N) benchmark validation
Reproduces dataset-statistics figures (confirm data packaging) and AlexNet/MPNet
encoding (confirm methodology), against the paper (doi:10.1038/s41593-026-02322-z).
Figures: 1f, 2f, 2g, 3d, 3e, 5.

In [1]:
import os, glob, re, json, warnings
import numpy as np, pandas as pd, scipy.io as sio, h5py, xarray as xr
from scipy.stats import pearsonr
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

DATA = os.environ.get("VAL_DATA", "/home/ec2-user/val")     # staged inputs
OUT = os.environ.get("VAL_OUT", "/home/ec2-user/val/figs"); os.makedirs(OUT, exist_ok=True)
TEMPORAL_NC = f"{DATA}/Li2026_temporal_assembly.nc"
PROC = f"{DATA}/Processed"
SUMMARY = {}

## Load per-unit table from Processed (response_best NSD images + region/patch/reliability)

In [2]:
area = pd.read_excel(f"{DATA}/exclude_area.xls", engine="xlrd"); area.columns = [c.strip() for c in area.columns]
sess_domain = {s: ("EVC" if set(g.Area.astype(str).str.strip().str.upper()) == {"EVC"} else "IT")
               for s, g in area.groupby("SesIdx")}
UT = {1: "SU", 2: "MUA", 3: "NonSom"}
def assign(pos, ses):
    for _, r in area[area.SesIdx == ses].iterrows():
        if r.y1 < pos < r.y2:
            a = str(r.AREALABEL).strip()
            if sess_domain[ses] == "EVC":
                for v in ("V1", "V2", "V4"):
                    if a.startswith(v): return v, v
                return "EVC-other", a
            return "IT", a
    return ("IT", "IT-other") if sess_domain[ses] == "IT" else ("EVC-other", "none")

resp, region, patch, rel, utype = [], [], [], [], []
for f in sorted(glob.glob(f"{PROC}/Processed_ses*.mat")):
    ses = int(re.search(r"ses(\d+)", os.path.basename(f)).group(1))
    m = sio.loadmat(f, squeeze_me=True)
    rb = np.atleast_2d(np.atleast_1d(m["response_best"]).astype(np.float32))[:, :1000]
    pos = np.atleast_1d(m["pos"]).astype(float); rr = np.atleast_1d(m["reliability_best"]).astype(float)
    ut = np.atleast_1d(m["UnitType"]).astype(int)
    for u in range(rb.shape[0]):
        rg, pl = assign(pos[u], ses)
        resp.append(rb[u]); region.append(rg); patch.append(pl); rel.append(rr[u]); utype.append(UT.get(ut[u], "?"))
resp = np.array(resp); region = np.array(region); patch = np.array(patch); rel = np.array(rel); utype = np.array(utype)
print("units:", len(resp), "| reliable>0.4:", int((rel > 0.4).sum()))

units: 47503 | reliable>0.4: 35494


## Fig 1f — split-half reliability by region

In [3]:
order = ["V1", "V2", "V4", "IT"]
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.boxplot([rel[region == r] for r in order], tick_labels=order, showfliers=False)
ax.axhline(0.4, color="r", ls="--", lw=1)
ax.set_ylabel("split-half reliability"); ax.set_title("Fig 1f: reliability by region")
fig.tight_layout(); fig.savefig(f"{OUT}/fig1f_reliability.png", dpi=120); plt.close(fig)
SUMMARY["1f_reliability_median"] = {r: round(float(np.median(rel[region == r])), 3) for r in order}
SUMMARY["1f_n_reliable"] = {r: int((rel[region == r] > 0.4).sum()) for r in order}
print("Fig1f median reliability:", SUMMARY["1f_reliability_median"])
print("Fig1f reliable counts (paper IT=26700):", SUMMARY["1f_n_reliable"])

Fig1f median reliability: {'V1': nan, 'V2': 0.802, 'V4': nan, 'IT': nan}
Fig1f reliable counts (paper IT=26700): {'V1': 2556, 'V2': 2625, 'V4': 3559, 'IT': 26700}


## Fig 2f — cross-area similarity (Pearson r between mean response profiles)

In [4]:
rmask = rel > 0.4
def cat(p):
    for k, v in [("MB", 0), ("AB", 0), ("MF", 1), ("AF", 1), ("PF", 1), ("MO", 2), ("AO", 2),
                 ("CLC", 3), ("AMC", 3), ("LPP", 4), ("PITP", 4)]:
        if p.startswith(k): return v
    return 5
prof = {p: np.nanmean(resp[rmask & (patch == p)], axis=0) for p in pd.unique(patch[rmask])
        if (rmask & (patch == p)).sum() >= 20 and cat(p) < 5}
plist = sorted(prof, key=lambda p: (cat(p), p)); C = np.corrcoef(np.array([prof[p] for p in plist]))
fig, ax = plt.subplots(figsize=(7, 6)); im = ax.imshow(C, cmap="RdBu_r", vmin=-0.6, vmax=0.6)
ax.set_xticks(range(len(plist))); ax.set_xticklabels(plist, rotation=90, fontsize=6)
ax.set_yticks(range(len(plist))); ax.set_yticklabels(plist, fontsize=6)
ax.set_title("Fig 2f: cross-area similarity"); fig.colorbar(im, label="Pearson r")
fig.tight_layout(); fig.savefig(f"{OUT}/fig2f_crossarea.png", dpi=120); plt.close(fig)
same = [C[i, j] for i in range(len(plist)) for j in range(i+1, len(plist)) if cat(plist[i]) == cat(plist[j])]
cross = [C[i, j] for i in range(len(plist)) for j in range(i+1, len(plist)) if cat(plist[i]) != cat(plist[j])]
SUMMARY["2f_within_vs_cross_r"] = [round(float(np.mean(same)), 3), round(float(np.mean(cross)), 3)]
print(f"Fig2f within-category r={np.mean(same):.3f} vs cross-category r={np.mean(cross):.3f} (paper: within>>cross)")

Fig2f within-category r=0.377 vs cross-category r=0.024 (paper: within>>cross)


## Fig 2g — noise & signal covariance (example session, SUs)

In [5]:
gf = f"{DATA}/GoodUnit_ses01.mat"
with h5py.File(gf, "r") as f:
    gus = f["GoodUnitStrc"]; pre = int(np.array(f["global_params"]["pre_onset"]).squeeze())
    nU = len(gus["Raster"])
    tvi = np.array(f["meta_data"]["trial_valid_idx"]).reshape(-1)
    dvi = np.array(f["meta_data"]["dataset_valid_idx"]).reshape(-1).astype(bool)
    timg = tvi[dvi].astype(int)
    _ur = gus["unittype"]
    ut = np.array([int(np.array(f[_ur[u][0]] if _ur.dtype == h5py.ref_dtype else _ur[u]).squeeze()) for u in range(nU)])
    rate = np.zeros((timg.size, nU), np.float32)
    for u in range(nU):
        ra = np.array(f[gus["Raster"][u][0]])
        rate[:, u] = ra[pre+70:pre+170].mean(0) * 1000
su = np.where(ut == 1)[0][:80]                       # SUs only (paper), cap 80 for a readable matrix
nsd = (timg >= 1) & (timg <= 1000)
R = rate[nsd][:, su]; img = timg[nsd]
imgs = np.unique(img)
sig_mean = np.array([R[img == i].mean(0) for i in imgs])          # (n_img, n_su) trial-averaged
signal_cov = np.cov(sig_mean.T)
noise_cov = np.mean([np.cov((R[img == i] - R[img == i].mean(0)).T) for i in imgs if (img == i).sum() > 1], axis=0)
iu = np.triu_indices_from(signal_cov, k=1)
ns_corr = pearsonr(noise_cov[iu], signal_cov[iu])[0]
fig, axs = plt.subplots(2, 1, figsize=(4.5, 8))
for axx, Mx, t in [(axs[0], noise_cov, "noise cov"), (axs[1], signal_cov, "signal cov")]:
    v = np.percentile(np.abs(Mx[iu]), 95); axx.imshow(Mx, cmap="RdBu_r", vmin=-v, vmax=v); axx.set_title(f"Fig 2g: {t}")
fig.suptitle(f"ses01 SUs (n={len(su)})  noise–signal r={ns_corr:.2f}")
fig.tight_layout(); fig.savefig(f"{OUT}/fig2g_covariance.png", dpi=120); plt.close(fig)
SUMMARY["2g_noise_signal_corr"] = round(float(ns_corr), 3)
print(f"Fig2g noise–signal covariance corr={ns_corr:.3f} (paper example Fig2g ~0.44)")

Fig2g noise–signal covariance corr=0.547 (paper example Fig2g ~0.44)


## Load temporal assembly (per-image PSTHs) for Fig 3d, 3e

In [6]:
T = xr.open_dataarray(TEMPORAL_NC); T.load()
treg = T["region"].values; trel = T["reliability"].values; tstart = T["time_bin_start"].values
print("temporal:", dict(T.sizes), "regions:", np.unique(treg))

temporal: {'presentation': 1000, 'neuroid': 47503, 'time_bin': 30} regions: ['EVC-other' 'IT' 'V1' 'V2' 'V4']


## Fig 3d — response-type clusters (k-means on per-unit PSTH) by region

In [7]:
from sklearn.cluster import KMeans
psth = T.mean("presentation").transpose("neuroid", "time_bin").values     # (neuroid, time)
keep = trel > 0.4
z = (psth[keep] - psth[keep].mean(1, keepdims=True)) / (psth[keep].std(1, keepdims=True) + 1e-9)
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(z)
lab = km.labels_; reg_k = np.where(np.isin(treg[keep], ["IT-other"]), "IT", treg[keep])
# order clusters by peak time for consistent naming
corder = np.argsort([tstart[np.argmax(km.cluster_centers_[c])] for c in range(3)])
remap = {c: i for i, c in enumerate(corder)}; lab = np.array([remap[l] for l in lab])
fig, axs = plt.subplots(1, 2, figsize=(10, 3.5))
for c in range(3):
    axs[0].plot(tstart, z[lab == c].mean(0), label=f"cluster {c+1} (n={int((lab==c).sum())})")
axs[0].set_title("Fig 3a: cluster mean PSTH"); axs[0].set_xlabel("ms"); axs[0].legend(fontsize=7)
regs = ["V1", "V2", "V4", "IT"]; bottom = np.zeros(len(regs))
for c in range(3):
    frac = [np.mean(lab[reg_k == r] == c) if (reg_k == r).sum() else 0 for r in regs]
    axs[1].bar(regs, frac, bottom=bottom, label=f"cluster {c+1}"); bottom += frac
axs[1].set_title("Fig 3d: response-type distribution by region"); axs[1].set_ylabel("fraction"); axs[1].legend(fontsize=7)
fig.tight_layout(); fig.savefig(f"{OUT}/fig3d_clusters.png", dpi=120); plt.close(fig)
SUMMARY["3d_cluster_fraction_by_region"] = {r: [round(float(np.mean(lab[reg_k == r] == c)), 2) for c in range(3)] for r in regs}
print("Fig3d cluster fractions by region:", SUMMARY["3d_cluster_fraction_by_region"])
print("(paper Fig3d: all regions contain all 3 response types in varying ratios)")

Fig3d cluster fractions by region: {'V1': [0.89, 0.05, 0.05], 'V2': [0.95, 0.04, 0.02], 'V4': [0.82, 0.12, 0.06], 'IT': [0.17, 0.53, 0.3]}
(paper Fig3d: all regions contain all 3 response types in varying ratios)


## Fig 3e — population RSA across time (IT)

In [8]:
itm = (np.isin(treg, ["IT", "IT-other"])) & (trel > 0.4)
Tit = T[{"neuroid": itm}].transpose("time_bin", "presentation", "neuroid").values   # (time, img, neuroid)
nb = Tit.shape[0]; rdms = []
for t in range(nb):
    X = Tit[t]; Xc = X - X.mean(1, keepdims=True)
    cc = np.corrcoef(Xc)                       # image x image
    rdms.append(1 - cc[np.triu_indices(cc.shape[0], 1)])
rdms = np.array(rdms)
S = np.corrcoef(rdms)                          # time x time RDM similarity
fig, ax = plt.subplots(figsize=(5, 4)); im = ax.imshow(S, origin="lower", cmap="viridis",
    extent=[tstart[0], tstart[-1], tstart[0], tstart[-1]])
ax.contour(S, levels=[0.3, 0.5], colors="w", extent=[tstart[0], tstart[-1], tstart[0], tstart[-1]])
ax.set_title("Fig 3e: population RSA across time (IT)"); ax.set_xlabel("ms"); ax.set_ylabel("ms")
fig.colorbar(im, label="RDM corr"); fig.tight_layout(); fig.savefig(f"{OUT}/fig3e_rsa_time.png", dpi=120); plt.close(fig)
SUMMARY["3e_rsa_diag_offdiag"] = [round(float(np.mean(np.diag(S, 1))), 3), round(float(S[5, nb-5]), 3)]
print(f"Fig3e RSA near-diagonal r={np.mean(np.diag(S,1)):.3f}, early-vs-late r={S[5,nb-5]:.3f} (paper: high near diag, structured)")

Fig3e RSA near-diagonal r=0.889, early-vs-late r=0.258 (paper: high near diag, structured)


## Fig 5 — AlexNet (FC6) + MPNet encoding accuracy (our models), per region

In [9]:
import torch, torchvision
from torchvision import transforms
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold

stim = pd.read_csv(f"{DATA}/Li2026_stimulus_set.csv").sort_values("tn_index")  # tn_index 1..1000 == resp columns
STIMDIR = f"{DATA}/stimuli"
tf = transforms.Compose([transforms.Resize(224), transforms.CenterCrop(224), transforms.ToTensor(),
                         transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
net = torchvision.models.alexnet(weights=torchvision.models.AlexNet_Weights.IMAGENET1K_V1).eval()
feat = {}
hook = net.classifier[2].register_forward_hook(lambda m, i, o: feat.__setitem__("fc6", i[0].detach()))  # input to ReLU after fc6
imgs_t = []
for sid in stim.stimulus_id:
    imgs_t.append(tf(Image.open(f"{STIMDIR}/{sid}.png").convert("RGB")))
X_alex = []
with torch.no_grad():
    for i in range(0, len(imgs_t), 100):
        net(torch.stack(imgs_t[i:i+100])); X_alex.append(feat["fc6"].numpy())
X_alex = np.concatenate(X_alex)            # (1000, 4096)
print("alexnet FC6:", X_alex.shape)

# MPNet semantic features from COCO captions (averaged over 5 captions/image)
def load_captions():
    info = pd.read_csv(f"{DATA}/nsd_stim_info_merged.csv")
    coco2split = dict(zip(info["cocoId"], info.get("cocoSplit", pd.Series(["train2017"]*len(info)))))
    caps = {}
    for sp in ["train2017", "val2017"]:
        p = f"{DATA}/captions_{sp}.json"
        if os.path.exists(p):
            j = json.load(open(p))
            for a in j["annotations"]:
                caps.setdefault(a["image_id"], []).append(a["caption"])
    return caps
X_mpnet = None
try:
    from sentence_transformers import SentenceTransformer
    caps = load_captions()
    if caps:
        mp = SentenceTransformer("all-mpnet-base-v2")
        embs = []
        for cid in stim.coco_id:
            cs = caps.get(int(cid), [""])[:5]
            embs.append(mp.encode(cs).mean(0))
        X_mpnet = np.array(embs)             # (1000, 768)
        print("mpnet:", X_mpnet.shape)
except Exception as e:
    print("MPNet skipped:", e)

def encode(Xfeat, Y, n_pca=100, n_pls=25, seed=0):
    """per-unit encoding accuracy (Pearson r), 10-fold CV. Xfeat:(img,feat) Y:(img,unit)."""
    Xp = PCA(n_components=min(n_pca, Xfeat.shape[1]), random_state=seed).fit_transform(
        (Xfeat - Xfeat.mean(0)) / (Xfeat.std(0) + 1e-9))
    pred = np.zeros_like(Y)
    for tr, te in KFold(10, shuffle=True, random_state=seed).split(Xp):
        pls = PLSRegression(n_components=min(n_pls, Xp.shape[1])).fit(Xp[tr], Y[tr])
        pred[te] = pls.predict(Xp[te])
    return np.array([pearsonr(Y[:, u], pred[:, u])[0] if Y[:, u].std() > 0 else np.nan for u in range(Y.shape[1])])

enc = {}
for r in order:
    mm = (region == r) & (rel > 0.4)
    Y = resp[mm].T                          # (1000, n_units)
    enc[(r, "alex")] = encode(X_alex, Y)
    if X_mpnet is not None:
        enc[(r, "mpnet")] = encode(X_mpnet, Y)
SUMMARY["5_alexnet_encoding_median_r"] = {r: round(float(np.nanmedian(enc[(r, "alex")])), 3) for r in order}
print("Fig5 AlexNet encoding median r by region:", SUMMARY["5_alexnet_encoding_median_r"])
if X_mpnet is not None:
    SUMMARY["5_mpnet_encoding_median_r"] = {r: round(float(np.nanmedian(enc[(r, "mpnet")])), 3) for r in order}
    # LVR: Deming slope of alexnet vs mpnet per-unit r (IT)
    a = enc[("IT", "alex")]; mp_ = enc[("IT", "mpnet")]; v = np.isfinite(a) & np.isfinite(mp_)
    # Deming (orthogonal) regression slope, delta=1
    x, y = mp_[v], a[v]; sxx = np.var(x); syy = np.var(y); sxy = np.cov(x, y)[0, 1]
    slope = (syy - sxx + np.sqrt((syy - sxx)**2 + 4*sxy**2)) / (2*sxy)
    SUMMARY["5_LVR_IT"] = round(float(1/slope), 3)   # visual/language ratio orientation
    print("Fig5 MPNet median r:", SUMMARY["5_mpnet_encoding_median_r"])
    print(f"Fig5 LVR (IT) ~= {SUMMARY['5_LVR_IT']} (paper LVR=0.74, <1 => more visual than semantic)")

# encoding accuracy distribution figure
fig, ax = plt.subplots(figsize=(6, 3.5))
for r in order:
    ax.hist(enc[(r, "alex")][np.isfinite(enc[(r, "alex")])], bins=40, histtype="step", label=f"{r} AlexNet")
ax.set_xlabel("per-unit encoding accuracy (Pearson r)"); ax.set_title("Fig 5: AlexNet encoding accuracy")
ax.legend(fontsize=7); fig.tight_layout(); fig.savefig(f"{OUT}/fig5_encoding.png", dpi=120); plt.close(fig)

Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /home/ec2-user/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


  0%|          | 0.00/233M [00:00<?, ?B/s]

 24%|██▍       | 56.4M/233M [00:00<00:00, 590MB/s]

 49%|████▊     | 113M/233M [00:00<00:00, 594MB/s] 

 73%|███████▎  | 170M/233M [00:00<00:00, 595MB/s]

...

alexnet FC6: (1000, 4096)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11430.81it/s]

mpnet: (1000, 768)


Fig5 AlexNet encoding median r by region: {'V1': 0.299, 'V2': 0.248, 'V4': 0.31, 'IT': 0.331}
Fig5 MPNet median r: {'V1': 0.145, 'V2': 0.106, 'V4': 0.168, 'IT': 0.236}
Fig5 LVR (IT) ~= 0.93 (paper LVR=0.74, <1 => more visual than semantic)


In [10]:
json.dump(SUMMARY, open(f"{OUT}/validation_summary.json", "w"), indent=2)
print("\n=== VALIDATION SUMMARY ==="); print(json.dumps(SUMMARY, indent=2))
print("VALIDATION_DONE")


=== VALIDATION SUMMARY ===
{
  "1f_reliability_median": {
    "V1": NaN,
    "V2": 0.802,
    "V4": NaN,
    "IT": NaN
  },
  "1f_n_reliable": {
    "V1": 2556,
    "V2": 2625,
    "V4": 3559,
    "IT": 26700
  },
  "2f_within_vs_cross_r": [
    0.377,
    0.024
  ],
  "2g_noise_signal_corr": 0.547,
  "3d_cluster_fraction_by_region": {
    "V1": [
      0.89,
      0.05,
      0.05
    ],
    "V2": [
      0.95,
      0.04,
      0.02
    ],
    "V4": [
      0.82,
      0.12,
      0.06
    ],
    "IT": [
      0.17,
      0.53,
      0.3
    ]
  },
  "3e_rsa_diag_offdiag": [
    0.889,
    0.258
  ],
  "5_alexnet_encoding_median_r": {
    "V1": 0.299,
    "V2": 0.248,
    "V4": 0.31,
    "IT": 0.331
  },
  "5_mpnet_encoding_median_r": {
    "V1": 0.145,
    "V2": 0.106,
    "V4": 0.168,
    "IT": 0.236
  },
  "5_LVR_IT": 0.93
}
VALIDATION_DONE
